In [ ]:
import numpy as np
import pickle
import sys
sys.path.append("..")
from scripts import indices

In [ ]:
f = open('../Output/Determ/rand_nor.pckl', 'rb')
A_nor = pickle.load(f)
f.close()

In [ ]:
A_nor.keys()

### Efficiency

In [ ]:
ave_eff = {}

In [ ]:
for p_in in A_nor.keys():
    print(p_in)
    ave_eff[p_in] = {}
    for p in A_nor[p_in].keys():
        print(p)
        ave_eff[p_in][p] = {}
        for k in A_nor[p_in][p].keys():
            print(k)
            ave_eff[p_in][p][k] = {}
            for ind in A_nor[p_in][p][k].keys():
                ave_eff[p_in][p][k][ind] = indices.average_efficiency(A_nor[p_in][p][k][ind])

In [ ]:
fname = '../Output/Determ/rand_eff_stb.pckl'
f = open(fname, 'wb')
pickle.dump(ave_eff, f)
f.close()

### Connectivity and path lengths

In [ ]:
connected_pairs = {}

In [ ]:
for p_in in A_nor.keys():
    print(p_in)
    connected_pairs[p_in] = {}
    for p in A_nor[p_in].keys():
        print(p)
        connected_pairs[p_in][p] = {}
        for k in A_nor[p_in][p].keys():
            print(k)
            connected_pairs[p_in][p][k] = {}
            for ind in A_nor[p_in][p][k].keys():
                connected_pairs[p_in][p][k][ind], path_length = indices.path_length_connectedness(A_nor[p_in][p][k][ind])

In [ ]:
fname = '../Output/Determ/rand_cp_stb.pckl'
f = open(fname, 'wb')
pickle.dump(connected_pairs, f)
f.close()

### number of convergent and divergent hubs

In [ ]:
binary_flag = True; thresh = 15

In [ ]:
Hubs = {}
Hubs['in'] = {}; Hubs['out'] = {}

In [ ]:
for p_in in A_nor.keys():
    Hubs['in'][p_in] = {}
    Hubs['out'][p_in] = {}
    for p in A_nor[p_in].keys():
        Hubs['in'][p_in][p] = {}
        Hubs['out'][p_in][p] = {}
        for k in A_nor[p_in][p].keys():
            Hubs['in'][p_in][p][k] = {}
            Hubs['out'][p_in][p][k] = {}
            for ind in A_nor[p_in][p][k].keys():
                Hubs['in'][p_in][p][k][ind] = indices.hub_number(A_nor[p_in][p][k][ind],thresh,binary_flag, axisUsed=1)
                Hubs['out'][p_in][p][k][ind] = indices.hub_number(A_nor[p_in][p][k][ind],thresh,binary_flag, axisUsed=0)

In [ ]:
fname = '../Output/Determ/rand_hubs_'+str(thresh)+'_stb.pckl'
f = open(fname, 'wb')
pickle.dump(Hubs, f)
f.close()

### get cd-units

In [ ]:
fname = '../Output/Determ/rand_cp_stb.pckl'
f = open(fname, 'rb')
connected_pairs = pickle.load(f)
f.close()

cd_pairs = {}; thresh_list = [13, 14, 15]; p_in = 0.5

In [ ]:
for thresh in thresh_list:
    cd_pairs[thresh] = {}
    for p in A_nor[p_in].keys():
        cd_pairs[thresh][p] = {}
        for k in A_nor[p_in][p].keys():
            cd_pairs[thresh][p][k] = {}
            for ind in A_nor[p_in][p][k].keys():
                cd_pairs[thresh][p][k][ind] = indices.cd_pairs(A_nor[p_in][p][k][ind], connected_pairs[p_in][p][k][ind], thresh)

In [ ]:
fname = '../Output/Determ/rand_cd_pairs_stb.pckl'
f = open(fname, 'wb')
pickle.dump(cd_pairs, f)
f.close()

### Stability of cd-units

In [ ]:
f = open('../Output/Determ/rand_cd_pairs_stb.pckl', 'rb')
cd_pairs = pickle.load(f)
f.close()

In [ ]:
num_config = {}
p_list = list(map(lambda x: x/10,range(11)))
K = 10

In [ ]:
for thresh in cd_pairs.keys():
    num_config[thresh] = np.zeros((2,len(p_list)))
    for p_ind, p in enumerate(p_list):
        num_config_p = np.zeros(K)
        for k in range(K):
            num_config_temp = np.zeros(len(cd_pairs[thresh][p][k].keys()))
            for i, step in enumerate(cd_pairs[thresh][p][k].keys()):
                num_config_temp[i] = cd_pairs[thresh][p][k][step].shape[1]
            num_config_p[k] = np.mean(num_config_temp==0)
        num_config[thresh][0,p_ind] = np.mean(num_config_p)
        num_config[thresh][1,p_ind] = np.std(num_config_p)

In [ ]:
fname = '../Output/Determ/rand_num_config.pckl'
f = open(fname, 'wb')
pickle.dump(num_config, f)
f.close()

### number of units

In [ ]:
num_cd = {}

In [ ]:
step = 15000
for thresh in cd_pairs.keys():
    num_cd[thresh] = np.zeros((2,len(p_list)))
    for p_ind, p in enumerate(p_list):
        num_cd_p = np.zeros(K)
        for k in range(K):
            num_cd_p[k] = cd_pairs[thresh][p][k][step].shape[1]
        num_cd[thresh][0,p_ind] = np.mean(num_cd_p)
        num_cd[thresh][1,p_ind] = np.std(num_cd_p)

In [ ]:
fname = '../Output/Determ/rand_num_cd.pckl'
f = open(fname, 'wb')
pickle.dump(num_cd, f)
f.close()

### Number of source and target nodes, and their overlap

In [ ]:
f = open('../Output/Determ/rand_cp_stb.pckl', 'rb')
connected_pairs = pickle.load(f)
f.close()

f = open('../Output/Determ/rand_cd_pairs_stb.pckl', 'rb')
cd_pairs = pickle.load(f)
f.close()

In [ ]:
p_in = 0.5; K = 10; step = 15000; thresh_list = [13, 14, 15]
num_targ = {}; num_sour = {}; prop_overlap = {}

In [ ]:
for thresh in thresh_list:
    num_targ[thresh] = {}
    num_sour[thresh] = {}
    prop_overlap[thresh] = {}
    for p in A_nor[p_in].keys():
        num_targ[thresh][p] = np.zeros(K)
        num_sour[thresh][p] = np.zeros(K)
        prop_overlap[thresh][p] = np.zeros(K)
        for k in range(K):
            if cd_pairs[thresh][p][k][step].shape[1]>0:
                num_targ_temp, num_sour_temp, prop_overlap_temp = indices.sour_targ_number(cd_pairs[thresh][p][k][step], connected_pairs[p_in][p][k][step])
                num_targ[thresh][p][k] = np.mean(num_targ_temp)
                num_sour[thresh][p][k] = np.mean(num_sour_temp)  
                prop_overlap[thresh][p][k] = np.mean(prop_overlap_temp)
            else:
                num_targ[thresh][p][k] = np.nan
                num_sour[thresh][p][k] = np.nan
                prop_overlap[thresh][p][k] = np.nan

In [ ]:
fname = '../Output/Determ/rand_num_sour.pckl'
f = open(fname, 'wb')
pickle.dump(num_sour, f)
f.close()

fname = '../Output/Determ/rand_num_targ.pckl'
f = open(fname, 'wb')
pickle.dump(num_targ, f)
f.close()

fname = '../Output/Determ/rand_prop_ovl.pckl'
f = open(fname, 'wb')
pickle.dump(prop_overlap, f)
f.close()

### Size and density of the intermediate subgraphs

In [ ]:
f = open('../Output/Determ/rand_cd_pairs_stb.pckl', 'rb')
cd_pairs = pickle.load(f)
f.close()

In [ ]:
p_in = 0.5; K = 10; step = 15000
interm_size = {}; interm_density = {}; periph_density = {}

In [ ]:
for thresh in thresh_list:
    interm_size[thresh] = {}
    interm_density[thresh] = {}
    periph_density[thresh] = {}
    for p in A_nor[p_in].keys():
        interm_size[thresh][p] = {}
        interm_density[thresh][p] = {}
        periph_density[thresh][p] = {}
        for k in range(K):
            if cd_pairs[thresh][p][k][step].shape[1]>0:
                interm_size[thresh][p][k],interm_density[thresh][p][k], periph_density[thresh][p][k] = indices.intermediate_subgraphs(A_nor[p_in][p][k][step], cd_pairs[thresh][p][k][step])
            else:
                interm_size[thresh][p][k] = np.nan
                interm_density[thresh][p][k] = np.nan
                periph_density[thresh][p][k] = np.nan

In [ ]:
fname = '../Output/Determ/rand_interm_size.pckl'
f = open(fname, 'wb')
pickle.dump(interm_size, f)
f.close()

fname = '../Output/Determ/rand_interm_density.pckl'
f = open(fname, 'wb')
pickle.dump(interm_density, f)
f.close()

fname = '../Output/Determ/rand_periph_density.pckl'
f = open(fname, 'wb')
pickle.dump(periph_density, f)
f.close()